___
# <font color= #003366> **Primer Examen Parcial** </font>
- <Strong> Integrantes: </Strong>  <font color="blue">`Clara Paola Aguilar Casillas, Roi Jared Flores Garza Stone, Mónica Ibarra Herrera, Diana Denise Valdivia Vargas` </font>
- <Strong> Materia: </Strong>  <font color="blue">`Modelos No Lineales Para Pronósticos` </font>

<div style="display: flex; align-items: center;">
    <div style="flex: 1;">
        <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="300">
    </div>
</div>

___

## <font color= #003366> **Objetivo** </font>

El objetivo del examen es desarrollar un modelo SARIMAX usando información obtenida mediante la API del Banco de México, con el fin de analizar y predecir el comportamiento del tipo de cambio de peso mexicano frente al dólar estadounidense.

Se busca identificar la estructura de la serie, evaluar su estacionariedad, estimar los parámetros óptimos del modelo y generar pronósticos diarios para el periodo del 5 al 13 de marzo de 2026.

Finalmente, se evaluará qué tan bien el puede capturar el modelo la dinámica del tipo de cambio.

## <font color= #003366> **Descripción de la Serie de Tiempo** </font>

La serie de tiempo que se usará en el examen corresponde al Tipo de Cambio FIX (peso mexicano – dólar estadounidense) publicado por el Banco de México.

Las características principales de la serie son:
- Nombre: Tipo de cambio FIX (MXN/USD)
- Fuente: API del Banco de México (SIE)
- Clave de serie: SF43718
- Frecuencia: Diaria
- Descripción: Indicador financiero publicado por Banxico que refleja el tipo de cambio peso mexicano - dólar estadounidense

## <font color= #003366> **Librerías** </font>

In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, acf, pacf, kpss
from scipy.stats import boxcox
from scipy.special import inv_boxcox
import plotly.graph_objects as go
import plotly.io as pio
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
pio.renderers.default = "vscode"

## <font color= #003366> **Conectar con la API** </font>

In [3]:
token = "c08c91a75598ee1e94c2081fb597314804b58642f10913c1e53f81c258ba5f17"

# Serie FIX
serie = "SF63528"

url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie}/datos"

headers = {
    "Bmx-Token": token
}

response = requests.get(url, headers=headers)

data = response.json()

In [ ]:
datos = data['bmx']['series'][0]['datos']

df = pd.DataFrame(datos)

df.columns = ['fecha', 'tipo_cambio']

df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')

df['tipo_cambio'] = pd.to_numeric(df['tipo_cambio'], errors='coerce')

df = df.sort_values('fecha')

df.set_index('fecha', inplace=True)
df

,tipo_cambio
fecha,
1954-04-19,0.0125
1954-04-20,0.0125
1954-04-21,0.0125
1954-04-22,0.0125
1954-04-23,0.0125
...,...
2026-02-26,17.2563
2026-02-27,17.2193
2026-03-02,17.3485


In [5]:
df_complete = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
missing_dates = df_complete.difference(df.index)

print("Número de fechas faltantes:", len(missing_dates))
missing_dates[:10]

Número de fechas faltantes: 7500


DatetimeIndex(['1954-04-24', '1954-04-25', '1954-05-01', '1954-05-02',
               '1954-05-08', '1954-05-09', '1954-05-15', '1954-05-16',
               '1954-05-22', '1954-05-23'],
              dtype='datetime64[us]', freq=None)

In [6]:
df_daily = df.reindex(df_complete)
df_daily.index.name = "fecha"

df_daily.head(10)

,tipo_cambio
fecha,
1954-04-19,0.0125
1954-04-20,0.0125
1954-04-21,0.0125
1954-04-22,0.0125
1954-04-23,0.0125
1954-04-24,NaN
1954-04-25,NaN
1954-04-26,0.0125
1954-04-27,0.0125


In [7]:
df_daily['tipo_cambio'] = df_daily['tipo_cambio'].ffill()
df_daily

,tipo_cambio
fecha,
1954-04-19,0.0125
1954-04-20,0.0125
1954-04-21,0.0125
1954-04-22,0.0125
1954-04-23,0.0125
...,...
2026-02-28,17.2193
2026-03-01,17.2193
2026-03-02,17.3485


La serie del tipo de cambio FIX solo incluye registros de días hábiles, por lo que no tiene registros en fines de semana ni días inhábiles.

Con el fin de trabajar con una serie continua y poder generar pronósticos para el periodo solicitado, se reindexó la serie a frecuencia diaria completa y se rellenaron las fechas faltantes utilizando el método forward fill, replicando el último valor observado. Esto permite mantener consistencia en el modelado sin alterar significativamente la dinámica de la serie.

## <font color= #003366> **Visualización de la serie de tiempo** </font>
Debido a que el pronóstico es de 2 semanas aproximadamente y la serie esta desde 1950, recortaremos la serie a partir del año 2021, manteniendo la información mas relevante para el modelo:

In [8]:
df = df_daily[df_daily.index >= "2021-01-01"]

df_plot = df.reset_index()

fig = px.line(
    df_plot,
    x="fecha",
    y="tipo_cambio",
    title="Tipo de Cambio FIX MXN/USD (20 años)"
)

fig.show()

## <font color= #003366> &ensp;  **Pruebas de Estacionareidad** </font>
Un supuesto que se tiene al trabajar con modelos **SARIMA** es la estacionariedad, la cuál es la propiedad que mantiene media, varianza y autocovarianza constantes a lo largo del tiempo.

Para validar si una serie de tiempo es estacionaria, se realizan las siguientes pruebas:
- Augmented Dickey-Fuller (ADF)

In [9]:
def check_stationarity(series, title="Serie Original"):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'Estadístico ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"¿Es estacionaria? {'SÍ' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. Revisamos la serie original
check_stationarity(df['tipo_cambio'], "Tipo de Cambio Original")

# 2. Aplicamos Primera Diferencia (d=1)
df['diff_1'] = df['tipo_cambio'].diff()

# 3. Revisamos la serie diferenciada
check_stationarity(df['diff_1'], "Primera Diferencia (d=1)")

# Creamos una figura con 2 columnas (Subplots)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Serie Original (No Estacionaria)", "Serie Diferenciada (Estacionaria d=1)")
)

# Gráfico 1: Serie Original
fig.add_trace(
    go.Scatter(x=df['tipo_cambio'].index, y=df['tipo_cambio'], name='Original'),
    row=1, col=1
)

# Gráfico 2: Serie Diferenciada
fig.add_trace(
    go.Scatter(x=df['diff_1'].index, y=df['diff_1'], name='Diferenciada'),
    row=1, col=2
)

# Ajustes de diseño
fig.update_layout(
    title_text="Comparativa: Efecto de la Diferenciación",
    showlegend=False, # Ocultamos leyenda
    height=500
)

fig.show()

ADF Test: Tipo de Cambio Original
Estadístico ADF: -1.2926
p-value: 0.6326
¿Es estacionaria? NO

ADF Test: Primera Diferencia (d=1)
Estadístico ADF: -13.5399
p-value: 0.0000
¿Es estacionaria? SÍ



Con la prueba de Estacionareidad podemos ver que la serie original **no** es estacionaria.

Se realizó la primera defirenciación y al volver a realizar la prueba podemos visualizar que la serie ya es estacionaria

## <font color= #003366> **Gráficas de ACF y PACF** </font>
Para poder determinar el orden de nuestro modelo, es necesario realizar las gráficas de **ACF** y **PACF**:

In [10]:
ts_analysis = df['diff_1'].dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

fig.show()

Se eligió diff 1 ya que la serie presenta tendencia a lo largo del tiempo, lo que la hace no estacionaria. Al diferenciar con lag 1 (restar cada valor del día anterior), se elimina esa tendencia y se obtiene una serie estacionaria, condición necesaria para modelar con ARIMA.

Podemos ver que con nuestra serie diferenciada, parece haber rezagos significativos a partit de el rezago 7, el 14 y uno que otro más, estos serán relevantes e indican que el periodo estacional de la serie es de 7 días s=7.  Sin embargo, a nivel personal, se observa que los primeros rezagos no parecen ser claramente significativos ni muestran un patrón muy definido. Por esta razón, se decidió probar distintos modelos con valores pequeños de p y q, principalmente 0 y 1, para evaluar cuál combinación presenta un mejor desempeño en términos de ajuste y comportamiento de los residuos.

In [11]:
df['diff_7'] = df['tipo_cambio'].diff(7).dropna()
ts_analysis = df['diff_7'].dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

# Resaltar lags estacionales (7, 14, 21, 28, 35) con líneas verticales rojas tenues
for i in [7, 14, 21, 28, 35]:
    fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()

Se eligió diff 7 ya que los patrones tienden a repetirse cada semana — por ejemplo, el comportamiento de un lunes se parece al del lunes anterior. Esta estacionalidad semanal es común en este tipo de series, y diferenciar con lag 7 permite eliminarla para obtener una serie estacionaria.

Podemos ver el la pacf que parece que los primeros 2 rezagos cada 7 días muestran relevancia y estos se van haciendo menos relevantes a partir de la mitad del 3 bloque (como el lag 21) Viendo la acf, elegir un orden q > 5 es algo excesivo asi que mejor alternaremos entre diferentes combinaciones de valores para P y Q, principalmente entre 1 y 2, comparando sus métricas de ajuste, con el fin de identificar el modelo que mejor represente la dinámica estacional de la serie.

## <font color= #66b0b0> &ensp;  **SARIMA** </font>
Procederemos a realizar el modelado de SARIMA con un test de 15 días de nuestra serie de tiempo, similar al intervalo de prediccón del examen:

In [12]:
# Realizar train/test split de 10%
num_dias = 15

train = df.iloc[:-num_dias]
test = df.iloc[-num_dias:]

col_target = "tipo_cambio"

# Realizar modelo de SARIMA
model = SARIMAX(train[col_target],
                order=(0,0,1),
                seasonal_order=(2,1,1,7),
                trend='c')

results = model.fit(disp=False,method='powell')

# Predecimos n pasos hacia el futuro (donde n = tamaño del test)
forecast_object = results.get_forecast(steps=len(test))
forecast_vals = forecast_object.predicted_mean
conf_int = forecast_object.conf_int(alpha=0.05) # Intervalo del 95%

# Metricas de error
rmse = np.sqrt(mean_squared_error(test[col_target], forecast_vals))
mape = mean_absolute_percentage_error(test[col_target], forecast_vals)
mae = mean_absolute_error(test[col_target], forecast_vals)


print(f"\n--- Errores del modelo ---")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.4f}")
print(f"MAE: {mae:.2f}")

print(results.summary())

# Grafica
fig = go.Figure()

# Train
fig.add_trace(go.Scatter(
    x=train[-num_dias-20:][col_target].index, y=train[-num_dias-20:][col_target],
    mode='lines',
    name='Train',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

# Test
fig.add_trace(go.Scatter(
    x=test[col_target].index, y=test[col_target],
    name='Test',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6)
))

# Forecast
fig.add_trace(go.Scatter(
    x=test.index, y=forecast_vals,
    name='SARIMA',
    line=dict(color='#606E34', width=3, dash='dot')
))

# Intervalos de Confianza
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 0],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))
fig.add_trace(go.Scatter(
    x=conf_int.index, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), fill='tonexty',
    fillcolor='rgba(96, 110, 52, 0.2)',
    name='Int. Confianza 95%', hoverinfo='skip'
))

# Titulos
fig.update_layout(
    title=f'<b>Modelo SARIMA: Pronóstico de Tipo de Cambio MLB</b><br><sup>',
    xaxis_title='Fecha',
    yaxis_title='Tipo de cambio',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()


--- Errores del modelo ---
RMSE: 0.17
MAPE: 0.0051
MAE: 0.09
                                     SARIMAX Results                                     
Dep. Variable:                       tipo_cambio   No. Observations:                 1874
Model:             SARIMAX(0, 0, 1)x(2, 1, 1, 7)   Log Likelihood                 408.301
Date:                           Wed, 04 Mar 2026   AIC                           -804.602
Time:                                   18:09:06   BIC                           -771.409
Sample:                               01-01-2021   HQIC                          -792.372
                                    - 02-17-2026                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     -0.0080      0.005     -1.639      0.101      

Después de realizar la prueba de estacionariedad y analizar las gráficas de ACF y PACF tanto en la parte no estacional como en la estacional, se probaron distintas combinaciones de parámetros para comparar su desempeño. 

Durante este proceso se observó que utilizar una diferenciación estacional (D = 1) daba mejores resultados que aplicar una diferenciación regular (d = 1), ya que permitía capturar de manera más adecuada el patrón semanal del tipo de cambio. 

Con base en las métricas de ajuste y el comportamiento de los residuos, se concluyó que el modelo que mejor representa la dinámica de la serie es un **SARIMA(0,0,1)(2,1,1,7)**. Este modelo logra capturar tanto la dinámica de corto plazo como la estacionalidad semanal presente en la serie, por lo que fue seleccionado para generar los pronósticos del periodo solicitado.

El modelo SARIMA predice bien el comportamiento estable de la serie, pero no logra capturar el salto abrupto de principios de marzo, lo cual sugiere que ese movimiento fue impulsado por factores externos no recogidos en el histórico.

## <font color= #66b0b0> &ensp; **SARIMA Forecast** </font>

In [13]:
fecha_inicio = pd.Timestamp('2026-03-05')
fecha_fin = pd.Timestamp('2026-03-13')
forecast_index = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

# Modelo
col_target = "tipo_cambio"

model_sarima = SARIMAX(
    df[col_target],
    order=(0,0,1),
    seasonal_order=(2,1,1,7),
    trend="c"
)

results_sarima = model_sarima.fit(disp=False, method='powell')

# Forecast
forecast_obj = results_sarima.get_forecast(steps=len(forecast_index))
forecast_vals = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int(alpha=0.05)

print(results_sarima.summary())


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df[col_target].index[-30:], 
    y=df[col_target].values[-30:],
    mode='lines',
    name='Histórico',
    line=dict(color='rgba(100, 100, 100, 0.6)', width=1.5)
))

fig.add_trace(go.Scatter(
    x=forecast_index, 
    y=forecast_vals,
    name='Pronóstico SARIMA',
    line=dict(color='#606E34', width=3, dash='dot')
))

fig.add_trace(go.Scatter(
    x=conf_int.index, 
    y=conf_int.iloc[:,0],
    mode='lines', 
    line=dict(width=0), 
    showlegend=False,
    hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=conf_int.index, 
    y=conf_int.iloc[:,1],
    mode='lines', 
    line=dict(width=0), 
    fill='tonexty',
    fillcolor='rgba(96, 110, 52, 0.2)',
    name='Int. Confianza 95%',
    hoverinfo='skip'
))

fig.update_layout(
    title='<b>Pronóstico SARIMA: Tipo de Cambio 5-13 Marzo 2026</b>',
    xaxis_title='Fecha',
    yaxis_title='Tipo de cambio',
    legend=dict(x=0, y=1, bgcolor='rgba(255,255,255,0.8)'),
    hovermode="x unified"
)

fig.show()

                                     SARIMAX Results                                     
Dep. Variable:                       tipo_cambio   No. Observations:                 1889
Model:             SARIMAX(0, 0, 1)x(2, 1, 1, 7)   Log Likelihood                 415.736
Date:                           Wed, 04 Mar 2026   AIC                           -819.471
Time:                                   18:09:29   BIC                           -786.231
Sample:                               01-01-2021   HQIC                          -807.229
                                    - 03-04-2026                                         
Covariance Type:                             opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     -0.0077      0.005     -1.611      0.107      -0.017       0.002
ma.L1          0.7048      0.015     46.117

El modelo SARIMA proyecta el tipo de cambio en un rango de 17.20 a 17.60 para la primera quincena de marzo de 2026, con mayor incertidumbre hacia el final del período. El pico proyectado alrededor del 10 de marzo refleja el patrón estacional semanal aprendido del comportamiento reciente.

In [21]:
df_forecast = pd.DataFrame({
    "Pronostico": forecast_vals,
    "Limite_inferior": conf_int.iloc[:, 0],
    "Limite_superior": conf_int.iloc[:, 1]
})

df_forecast.index = forecast_index

df_forecast.to_excel("pronostico_sarima_1.xlsx", index=True, index_label="Fecha")

## <font color= #003366> **Resultados Obtenidos** </font>

- **Errores del test:** obtuvimos las siguientes métricas de error en nuestro modelo SARIMA final:
    - **MAE**: 0.09
    - **RMSE**: 0.17
    - **MAPE**: 0.0051

- A través del análisis de las funciones de autocorrelación (ACF) y autocorrelación parcial (PACF), se seleccionaron los órdenes óptimos para los componentes Autorregresivos (AR) y de Medias Móviles (MA).
    - (0,0,1) (2,1,1,7)

- **Coeficientws estadísitcamente significantes**:
    - ar.S.l7
    - ma.S.l7
    - sigma2

- **Coeficientes estadísiticamente no significantes**:
    - interecept
    - ar.S.l14

## <font color= #003366> **Conclusiones** </font>

- **Eficacia de la Metodología**: El flujo de trabajo (desde la extracción vía API hasta el diagnóstico estadístico) demostró ser robusto para capturar la inercia y los patrones de dependencia lineal en el tipo de cambio MXN/USD. <br>

- **Importancia de la Diferenciación**: Se concluye que el tipo de cambio es una serie no estacionaria, por lo que el uso de modelos que incluyan el parámetro $d$ (como ARIMA/SARIMA) es fundamental para evitar malas regresiones. <br>
- **Valor de la Incertidumbre**: La inclusión de intervalos de confianza en los resultados finales subraya que, aunque existe una tendencia proyectada, el mercado cambiario está sujeto a una volatilidad que el modelo representa como un margen de error probable. <br>
- **Aplicación Práctic**a: El ejercicio cumple con el objetivo académico de aplicar modelos lineales y no lineales para la toma de decisiones financieras, permitiendo anticipar escenarios posibles para la moneda nacional en el corto plazo (aunque el mercado siempre es volátil y una variable externa puede ocasionar un mal pronóstico).